# M2 on CREMA-D: Qwen2.5-Omni-7B zero-shot (Kaggle, 2× T4)

Label scoring of all CREMA-D clips under the three conditions the human raters saw: audio-visual (`AV`), audio only
(`A`), video only (`V`). No transcript, since the raters did not read the sentence. Six labels (CREMA-D's).
Same scoring as the CH-SIMS runs: logits of each label's first token + log-sum-exp over the vocabulary, written to
`/kaggle/working/cremad_qwen/scores.jsonl`.

**Running it**
1. *Accelerator*: **GPU T4 x2**; *Internet*: **on**; *Add input*: the output of `prep_cremad.ipynb` (`cremad_mm/`).
2. First set `LIMIT = 10` and run all interactively; send the timing lines and the quick-check table.
3. Full run: `LIMIT = None`, **Save Version → Save & Run All (Commit)**. It stops after `TIME_BUDGET_H`;
   to continue, add this notebook's output as an input and commit again (finished items are skipped).

In [ ]:
# Versions these notebooks were written against (transformers 5.x API: `dtype=`, thinker class).
%pip install -q "transformers==5.20.1" "qwen-omni-utils==0.0.9" audioread librosa

In [ ]:
import glob
import json
import os
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-Omni-7B")  # env override: local tests with a tiny model
LOAD_IN_4BIT = False         # True: 4-bit NF4 weights (needs bitsandbytes), if fp16 does not fit
MAX_MEMORY = {0: "12GiB", 1: "14GiB"}  # leave headroom on GPU 0 for activations
FRAME_MAX_PIXELS = 448 * 252           # per-frame pixel cap passed to the Qwen video processor
DRY_RUN = os.environ.get("DRY_RUN") == "1"  # build inputs only, no model (for local checks)
SEED = 0

# Folder made by analysis/preprocess_chsims.py; found automatically under /kaggle/input.
DATA_DIR = os.environ.get("CHSIMS_MM_DIR")
if DATA_DIR is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "manifest.csv"))]  # no ** : media folders make it slow
    assert hits, "No manifest.csv under /kaggle/input: attach the preprocessed dataset."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)

# Folder made by kaggle/prep_cremad.ipynb (analysis/preprocess_cremad.py); found automatically under /kaggle/input.
DATA_DIR = os.environ.get("CREMAD_MM_DIR")
if DATA_DIR is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "cremad_mm/manifest.csv"))]
    assert hits, "No cremad_mm/manifest.csv under /kaggle/input: attach the output of prep_cremad.ipynb."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)

TIME_BUDGET_H = 11.0         # stop starting new items after this many hours
LIMIT = None                 # e.g. 10 for a quick check: 10 test clips + 10 other clips
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/cremad_qwen"))
T_START = time.time()

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| out:", OUT_DIR, "| dry run:", DRY_RUN)
print("free disk on / (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}", torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), "GiB")

## Load the processor and the thinker model

In [ ]:
from qwen_omni_utils import process_mm_info
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
model = None
if not DRY_RUN:
    kwargs = dict(dtype=torch.float16, device_map="auto", max_memory=MAX_MEMORY, attn_implementation="sdpa")
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    t0 = time.time()
    model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
    print(f"loaded in {time.time() - t0:.0f} s")
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i} allocated after load: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB")

## Prompt

In [ ]:
LABELS = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]  # CREMA-D's six labels, as the raters saw them
DESCRIBE = {  # the three conditions the raters saw; no transcript (raters did not read the sentence)
    "AV": "the video frames and the audio track",
    "A": "only the audio track",
    "V": "only the video frames, without sound",
}
CONDITIONS = list(DESCRIBE)
ASK = f"Which emotion does the speaker express? Choose exactly one of: {', '.join(LABELS)}.\nAnswer with the label only."


def intro(condition):
    return f"You are given {DESCRIBE[condition]} of an actor saying a short sentence.\n"

SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
label_first_ids = [processor.tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(label_first_ids)) == len(LABELS), "two labels share a first token"


def build_messages(row, condition):
    content = []
    if "V" in condition:
        frames = sorted(glob.glob(str(DATA_DIR / row["frames_dir"] / "*.jpg")))
        content.append({"type": "video", "video": [f"file://{f}" for f in frames],
                        "sample_fps": float(row["sample_fps"]), "max_pixels": FRAME_MAX_PIXELS})
    if "A" in condition:
        content.append({"type": "audio", "audio": str(DATA_DIR / row["audio_path"])})
    content.append({"type": "text", "text": intro(condition) + ASK})
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content}]


def prepare(messages):
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    audios, images, videos, video_kwargs = process_mm_info(messages, use_audio_in_video=False, return_video_kwargs=True)
    fps = video_kwargs.pop("fps", [])  # qwen-omni-utils returns a per-video list; the processor wants one number
    if fps:
        video_kwargs["fps"] = float(fps[0])
    inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt",
                       padding=True, use_audio_in_video=False, **video_kwargs)
    if model is not None:
        inputs = {k: (v.to(model.device, dtype=model.dtype) if torch.is_floating_point(v) else v.to(model.device))
                  if torch.is_tensor(v) else v for k, v in inputs.items()}
    return inputs

## Clips

All clips with `status == ok` (7,438). `ds_label` (the audio-visual raters' majority label, ties to the first label)
is used only for the accuracy column of the quick check at the end.

In [ ]:
manifest = pd.read_csv(DATA_DIR / "manifest.csv", dtype={"video_id": str})
clips = manifest[manifest["status"] == "ok"].reset_index(drop=True)
clips["ds_label"] = np.array(LABELS)[clips[[f"av_{l}" for l in LABELS]].to_numpy().argmax(1)]
if LIMIT:
    clips = pd.concat([clips[clips["mode"] == "test"].head(LIMIT), clips[clips["mode"] != "test"].head(LIMIT)])
print(len(clips), "clips to score")

## Resume

Records are keyed by (model, uid, condition, task). Earlier records found under `/kaggle/input` or in
`OUT_DIR` are copied into this run's output and skipped below.

In [ ]:
SCORES = OUT_DIR / "scores.jsonl"


def key(rec):
    return (rec["model"], rec["uid"], rec["condition"], rec["task"])


sources = sorted(set(h for k in range(6) for h in glob.glob("/kaggle/input/" + "*/" * k + "cremad_qwen/scores.jsonl")) | ({str(SCORES)} if SCORES.exists() else set()))
records = {}
for src in sources:
    for line in open(src, encoding="utf-8"):
        if line.strip() and "error" not in (rec := json.loads(line)):
            records[key(rec)] = rec
with open(SCORES, "w", encoding="utf-8") as f:
    for rec in records.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
done = set(records)
print(f"{len(records)} finished records from {len(sources)} file(s)")


def out_of_time():
    return (time.time() - T_START) / 3600 > TIME_BUDGET_H

## Run

All clips under the first condition in `CONDITIONS` (the full one) first, then the others, so a run cut short
still has the main condition complete.
A clip that fails (for example out of memory) is saved with an `error` field and retried on the next run.

In [ ]:
@torch.inference_mode()
def run_score(row, condition):
    t0 = time.perf_counter()
    inputs = prepare(build_messages(row, condition))
    t1 = time.perf_counter()
    logits = model(**inputs, use_cache=False).logits[0, -1].float()
    rec = {"model": MODEL_ID, "uid": row["uid"], "condition": condition, "task": "score",
           "n_tokens": int(inputs["input_ids"].shape[1]), "prep_s": round(t1 - t0, 3),
           "gpu_s": round(time.perf_counter() - t1, 3), "finite": bool(torch.isfinite(logits).all()),
           "lse": float(torch.logsumexp(logits, dim=-1))}
    rec.update({f"logit_{l}": float(v) for l, v in zip(LABELS, logits[label_first_ids].tolist())})
    return rec


jobs = [(row, c) for c in CONDITIONS for _, row in clips.iterrows()]
todo = [j for j in jobs if (MODEL_ID, j[0]["uid"], j[1], "score") not in done]
print(f"{len(jobs)} items, {len(jobs) - len(todo)} already done, {len(todo)} to run")

n_done, stopped_early, t_run = 0, False, time.time()
for row, condition in todo:
    if out_of_time():
        stopped_early = True
        break
    try:
        rec = run_score(row, condition)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        rec = {"model": MODEL_ID, "uid": row["uid"], "condition": condition, "task": "score", "error": "oom"}
    with open(SCORES, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    n_done += 1
    if n_done in (20, 100) or n_done % 500 == 0:
        sec = (time.time() - t_run) / n_done
        peak = max([torch.cuda.max_memory_allocated(d) / 2**30 for d in range(torch.cuda.device_count())] or [0])
        print(f"{n_done}/{len(todo)} | {sec:.2f} s/item, about {(len(todo) - n_done) * sec / 3600:.1f} h left | "
              f"peak {peak:.1f} GiB | last: {condition}")

print(f"finished {n_done} items this run", "(stopped at the time budget: commit again to continue)" if stopped_early else "")

## Progress and quick checks

In [ ]:
s = pd.DataFrame([json.loads(l) for l in open(SCORES, encoding="utf-8") if l.strip()])
ok = s[s.get("error", pd.Series(index=s.index, dtype=object)).isna()] if len(s) else s
print(f"{len(ok)} / {len(clips) * len(CONDITIONS)} items done |", "OOM:", int((s.get("error") == "oom").sum()) if "error" in s else 0)
if len(ok):
    p = np.exp(ok[[f"logit_{l}" for l in LABELS]].to_numpy() - ok[["lse"]].to_numpy())
    ok = ok.assign(label_mass=p.sum(1), pred=np.array(LABELS)[p.argmax(1)]).merge(clips[["uid", "ds_label"]], on="uid")
    print(ok.groupby("condition").agg(n=("uid", "size"), label_mass=("label_mass", "mean"),
                                      accuracy=("pred", lambda x: (x == ok.loc[x.index, "ds_label"]).mean()),
                                      nonfinite=("finite", lambda x: int((~x.astype(bool)).sum())),
                                      gpu_s=("gpu_s", "mean"), prep_s=("prep_s", "mean")).round(3).to_string())